In [ ]:
HarvardOxford.xml
 ↓
48 labels 추출
 ↓
Wikipedia 수집
 ↓
BrainInfo 수집
 ↓
raw_brain_regions.json
 ↓
GPT 요약
 ↓
brain_regions_knowledge.json
 ↓
FAISS
 ↓
RAG

In [12]:
import requests

def get_wikipedia_extract(title):

    headers = {
        "User-Agent": "BrainTumorRAG/1.0"
    }

    params = {
        "action": "query",
        "prop": "extracts",
        "explaintext": True,
        "format": "json",
        "titles": title
    }

    response = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params=params,
        headers=headers
    )

    pages = response.json()["query"]["pages"]

    page = next(iter(pages.values()))

    return page.get("extract", "")

In [13]:
text = get_wikipedia_extract("Insular cortex")

print(len(text))
print(text[:1500])

25028
The insular cortex (also insula and insular lobe) is a portion of the cerebral cortex folded deep within the lateral sulcus (the fissure separating the temporal lobe from the parietal and frontal lobes) within each hemisphere of the mammalian brain.
The insulae are believed to be involved in consciousness and play a role in diverse functions usually linked to emotion, interoception, or the regulation of the body's homeostasis. These functions include compassion, empathy, taste, perception, motor control, self-awareness, cognitive functioning, interpersonal relationships, and awareness of homeostatic emotions such as hunger, pain and fatigue. In relation to these, it is involved in psychopathology.
The insular cortex is divided by the central sulcus of the insula, into two parts: the anterior insula and the posterior insula in which more than a dozen field areas have been identified. The cortical area overlying the insula toward the lateral surface of the brain is the operculum (m

In [ ]:
from playwright.async_api import async_playwright
import asyncio

async def test():

    async with async_playwright() as p:

        browser = await p.chromium.launch(
            headless=False
        )

        page = await browser.new_page()

        await page.goto(
            "http://braininfo.rprc.washington.edu/searchByFunctionalModel"
        )

        await page.wait_for_timeout(5000)

        print(await page.title())

        input("확인 후 엔터")

        await browser.close()

await test()

Search By Functional Model - BrainInfo


In [6]:
from playwright.async_api import async_playwright

async def inspect():

    async with async_playwright() as p:

        browser = await p.chromium.launch(
            headless=False
        )

        page = await browser.new_page()

        await page.goto(
            "http://braininfo.rprc.washington.edu/searchByFunctionalModel"
        )

        await page.wait_for_timeout(10000)       

        text = await page.locator("body").inner_text()

        buttons = await page.locator("button").all_inner_texts()
        print(buttons)

        print(text[:3000])

        await browser.close()

await inspect()

['MENU', 'SEARCH BY NAME', '', 'SEARCH']
MENU
HOME
SEARCH BY NAME
Neural Structure
Functional Model
Relation / Feature
Authors of Sources
SEARCH BY ACRONYM
BRAIN ATLASES
ABOUT BRAININFO
COPYRIGHT
What functional model is of interest?

Examples: human brain, visual system, default network, etc.

Enter model name
SEARCH
BrainInfo
Copyright 1991-present
UW


In [40]:
from playwright.async_api import async_playwright

async def inspect_page():

    async with async_playwright() as p:

        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()

        await page.goto(
            "http://braininfo.rprc.washington.edu/searchByFunctionalModel"
        )
        
        await page.wait_for_timeout(5000)

        await page.fill("input", "Insular Cortex")

        await page.wait_for_timeout(1000)
        
        #print("before:", page.url)

        await page.keyboard.press("Enter")

        await page.wait_for_timeout(5000)

        # 검색결과 확인
        #body = await page.locator("body").inner_text()
        #print(body)

        # 검색결과에 따른 URL 확인
        #links = await page.locator("a").count()
        #print("links:", links)
        #for i in range(min(30, links)):
        #    href = await page.locator("a").nth(i).get_attribute("href")
        #    text = await page.locator("a").nth(i).inner_text()
        #    print(text, "=>", href)

        links = page.locator('a[href*="/brain-concept/"]')

        count = await links.count()

        for i in range(count):

            text = await links.nth(i).inner_text()

            if text.lower() == "insular cortex":
                href = await links.nth(i).get_attribute("href")
                break

        #print(href)
        await page.goto(
            "http://braininfo.rprc.washington.edu" + href
        )

        #print(page.url)
        #print(await page.title())

        html = await page.content()

        #body = await page.locator("body").inner_text()
        #print(body[:5000])
        # 본문만 추출
        description = await page.locator(
            'meta[name="description"]'
        ).get_attribute("content")

        print(description)
await inspect_page()

The term insular cortex refers to the set of cortical areas defined on the basis of internal structure that comprise the insula in the human and the macaque .…


In [104]:
# XML -> 영역 리스트 만들기
import xml.etree.ElementTree as ET

def load_harvard_regions(xml_path):

    tree = ET.parse(xml_path)
    root = tree.getroot()

    regions = []

    for label in root.findall(".//label"):

        regions.append({
            "index": int(label.attrib["index"]),
            "name": label.text.strip()
        })

    return regions


regions = load_harvard_regions(
    "/home/lhe09/projects/msd_brain/HarvardOxford-Cortical.xml"
)

print(len(regions))
print(regions[:5])

48
[{'index': 0, 'name': 'Frontal Pole'}, {'index': 1, 'name': 'Insular Cortex'}, {'index': 2, 'name': 'Superior Frontal Gyrus'}, {'index': 3, 'name': 'Middle Frontal Gyrus'}, {'index': 4, 'name': 'Inferior Frontal Gyrus, pars triangularis'}]


In [105]:
# Wikipidia에서 각 영역 정보 수집
import requests
from difflib import SequenceMatcher

def similarity(a, b):

    return SequenceMatcher(
        None,
        a.lower(),
        b.lower()
    ).ratio()

def get_wikipedia_text(region_name):

    try:

        search_url = (
            "https://en.wikipedia.org/w/api.php"
        )

        params = {
            "action": "query",
            "list": "search",
            "srsearch": region_name,
            "format": "json"
        }

        r = requests.get(
            search_url,
            params=params,
            headers={
                "User-Agent":
                "BrainAtlasBot/1.0"
            }
        )

        results = r.json()["query"]["search"]

        if len(results) == 0:
            return None

        title = results[0]["title"]

        # 동일한 검색어 없음
        #score = similarity(region_name,title)
        #print(f"{region_name} -> {title} ({score:.2f})")
        #if score < 0.6:
        #    print(f"Skip Wiki: {region_name}")
        #    return None

        best_score = 0
        best_title = None

        for result in results[:5]:

            title = result["title"]

            score = similarity(
                region_name,
                title
            )

            if score > best_score:

                best_score = score
                best_title = title
        if best_score < 0.75:
            return None
        
        print(
            f"{region_name} -> {best_title} ({best_score:.2f})"
        )

        extract_url = (
            "https://en.wikipedia.org/w/api.php"
        )

        params = {
            "action": "query",
            "prop": "extracts",
            "explaintext": True,
            "titles": best_title,
            "format": "json"
        }

        r = requests.get(
            extract_url,
            params=params,
            headers={
                "User-Agent":
                "BrainAtlasBot/1.0"
            }
        )

        pages = r.json()["query"]["pages"]

        page = next(iter(pages.values()))

        return {
            "title": best_title,
            "similarity": round(best_score, 2),
            "text": page.get("extract", "")
        }

    except Exception as e:

        print("Wiki Error:", region_name, e)

        return None

In [106]:
# BrainInfo에서 각 영역 정보 수집
from playwright.async_api import async_playwright
import asyncio

from difflib import SequenceMatcher

def similarity(a, b):

    return SequenceMatcher(
        None,
        a.lower(),
        b.lower()
    ).ratio()

async def get_braininfo(region_name):

    async with async_playwright() as p:

        browser = await p.chromium.launch(
            headless=True
        )
        page = await browser.new_page()
        await page.goto(
            "http://braininfo.rprc.washington.edu/searchByFunctionalModel"
        )

        await page.wait_for_timeout(5000)

        await page.fill(
            "input",
            region_name
        )
        
        await page.keyboard.press("Enter")
        await page.wait_for_timeout(3000)

        links = page.locator(
            'a[href*="/brain-concept/"]'
        )

        count = await links.count()

        if count == 0:

            await browser.close()
            return None

        best_score = 0
        best_href = None
        best_name = None

        for i in range(count):

            text = await links.nth(i).inner_text()

            score = similarity(
                region_name,
                text
            )

            if score > best_score:

                best_score = score

                best_name = text

                best_href = await links.nth(i).get_attribute(
                    "href"
                )
        print(
            f"{region_name} -> {best_name} ({best_score:.2f})"
        )

        if best_score < 0.55:

            await browser.close()

            return None
        
        url = (
            "http://braininfo.rprc.washington.edu"
            + best_href
        )

        matched_name = best_name
        
        await page.goto(url)

        await page.wait_for_timeout(3000)

        description = await page.locator(
            'meta[name="description"]'
        ).get_attribute("content")

        title = await page.title()

        await browser.close()

        return {
            "matched_term": matched_name,
            "url": url,
            "title": title,
            "description": description
        }

In [107]:
# raw JSON 생성(GPT까지 돌리고 저장하면 에러 수정이 길어질까봐)
import json
import asyncio
import re

# 예외로 검색이 안되는 것들이 너무 많아 예외로 alias 사전을 추가함
ALIAS = {

    # 기존
    "Frontal Orbital Cortex":
        "Orbitofrontal cortex",

    "Precuneous Cortex":
        "Precuneus",

    "Cuneal Cortex":
        "Cuneus",

    "Subcallosal Cortex":
        "Subcallosal area",

    "Juxtapositional Lobule Cortex (formerly Supplementary Motor Cortex)":
        "Supplementary motor area",

    "Heschl's Gyrus (includes H1 and H2)":
        "Heschl's gyrus",

    # 추가
    "Lateral Occipital Cortex, superior division":
        "Lateral occipital cortex",

    "Lateral Occipital Cortex, inferior division":
        "Lateral occipital cortex",

    "Frontal Medial Cortex":
        "Medial prefrontal cortex",

    "Frontal Orbital Cortex":
        "Orbitofrontal cortex",

    "Temporal Fusiform Cortex, anterior division":
        "Fusiform gyrus",

    "Temporal Fusiform Cortex, posterior division":
        "Fusiform gyrus",

    "Temporal Occipital Fusiform Cortex":
        "Fusiform gyrus",

    "Occipital Fusiform Gyrus":
        "Fusiform gyrus",

    "Frontal Operculum Cortex":
        "Frontal operculum",

    "Central Opercular Cortex":
        "Operculum",

    "Parietal Operculum Cortex":
        "Parietal operculum",

    "Supracalcarine Cortex":
        "Visual cortex",

    "Intracalcarine Cortex":
        "primary visual cortex",
        
    "Heschl's Gyrus (includes H1 and H2)":
        "transverse temporal gyrus",

    "Lateral Occipital Cortex, superior division":
        "occipital cortex",

    "Lateral Occipital Cortex, inferior division":
        "occipital cortex",

    "Paracingulate Gyrus":
        "paracingulate"

}


SPECIAL_MAPPING = {

    "Juxtapositional Lobule Cortex (formerly Supplementary Motor Cortex)":
        "Supplementary motor area",

    "Heschl's Gyrus (includes H1 and H2)":
        "Heschl's gyrus"
}

def normalize_region_name(region_name):

    name = region_name
    if name in ALIAS:
        return ALIAS[name]

    if name in SPECIAL_MAPPING:
        return SPECIAL_MAPPING[name]

    remove_terms = [
        ", anterior division",
        ", posterior division",
        ", temporooccipital part",
        ", superior division",
        ", inferior division",
        ", pars triangularis",
        ", pars opercularis",
    ]


    for term in remove_terms:
        name = name.replace(term, "")

    # 괄호 제거
    name = re.sub(
        r"\s*\(.*?\)",
        "",
        name
    )

    return name.strip()

async def build_raw_json():

    regions = load_harvard_regions(
        "/home/lhe09/projects/msd_brain/HarvardOxford-Cortical.xml"
    )

    all_data = []

    for region in regions:

        name = region["name"]

        print("Processing:", name)
        
        search_name = normalize_region_name(name)

        wiki = get_wikipedia_text(search_name)

        braininfo = await get_braininfo(search_name)

        all_data.append({

            "index": region["index"],

            "atlas_label": name,

            "search_name": search_name,

            "wikipedia": wiki,

            "braininfo": braininfo
        })

        with open(
            "raw_brain_regions.json",
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                all_data,
                f,
                indent=2,
                ensure_ascii=False
            )

    return all_data

In [108]:
# 실행
raw_data = await build_raw_json()

Processing: Frontal Pole
Frontal Pole -> Frontal lobe (0.83)
Frontal Pole -> frontal pole (1.00)
Processing: Insular Cortex
Insular Cortex -> Insular cortex (1.00)
Insular Cortex -> insular cortex (1.00)
Processing: Superior Frontal Gyrus
Superior Frontal Gyrus -> Superior frontal gyrus (1.00)
Superior Frontal Gyrus -> superior frontal gyrus (1.00)
Processing: Middle Frontal Gyrus
Middle Frontal Gyrus -> Middle frontal gyrus (1.00)
Middle Frontal Gyrus -> middle frontal gyrus (1.00)
Processing: Inferior Frontal Gyrus, pars triangularis
Inferior Frontal Gyrus -> Inferior frontal gyrus (1.00)
Inferior Frontal Gyrus -> inferior frontal gyrus (1.00)
Processing: Inferior Frontal Gyrus, pars opercularis
Inferior Frontal Gyrus -> Inferior frontal gyrus (1.00)
Inferior Frontal Gyrus -> inferior frontal gyrus (1.00)
Processing: Precentral Gyrus
Precentral Gyrus -> Precentral gyrus (1.00)
Precentral Gyrus -> precentral gyrus (1.00)
Processing: Temporal Pole
Temporal Pole -> temporal pole (1.00)


In [109]:
# 실패 목록
import json

with open(
    "raw_brain_regions.json",
    "r",
    encoding="utf-8"
) as f:

    raw_data = json.load(f)

for item in raw_data:

    if item["braininfo"] is None:

        print(item["atlas_label"])

In [103]:
for item in raw_data:

    if item["braininfo"] is None:

        print(
            item["atlas_label"],
            "->",
            item["search_name"]
        )

Intracalcarine Cortex -> Calcarine cortex
Paracingulate Gyrus -> paracigulate


In [ ]:
# BrainInfo에서 null  데이터 원인 파악
from playwright.async_api import async_playwright
import asyncio

from difflib import SequenceMatcher

def similarity(a, b):

    return SequenceMatcher(
        None,
        a.lower(),
        b.lower()
    ).ratio()

async def get_braininfo(region_name):

    async with async_playwright() as p:

        browser = await p.chromium.launch(
            headless=True
        )
        page = await browser.new_page()
        await page.goto(
            "http://braininfo.rprc.washington.edu/searchByFunctionalModel"
        )

        await page.wait_for_timeout(5000)

        await page.fill(
            "input",
            region_name
        )
        
        await page.keyboard.press("Enter")
        await page.wait_for_timeout(3000)

        links = page.locator(
            'a[href*="/brain-concept/"]'
        )

        count = await links.count()
        print(f"\nSEARCH: {region_name}")
        print(f"LINK COUNT: {count}")

        for i in range(min(count,10)):
            
            text = await links.nth(i).inner_text()

            print(text)

        if count == 0:

            await browser.close()
            return None

        best_score = 0
        best_href = None
        best_name = None

        for i in range(count):

            text = await links.nth(i).inner_text()

            score = similarity(
                region_name,
                text
            )

            if score > best_score:

                best_score = score

                best_name = text

                best_href = await links.nth(i).get_attribute(
                    "href"
                )
        print(
            f"{region_name} -> {best_name} ({best_score:.2f})"
        )

        if best_score < 0.55:

            await browser.close()

            return None
        
        url = (
            "http://braininfo.rprc.washington.edu"
            + best_href
        )

        matched_name = best_name
        
        await page.goto(url)

        await page.wait_for_timeout(3000)

        description = await page.locator(
            'meta[name="description"]'
        ).get_attribute("content")

        title = await page.title()

        await browser.close()

        return {
            "matched_term": matched_name,
            "url": url,
            "title": title,
            "description": description
        }

In [96]:
await get_braininfo("paracingulate")



SEARCH: paracingulate
LINK COUNT: 1
paracingulate sulcus
paracingulate -> paracingulate sulcus (0.79)


{'matched_term': 'paracingulate sulcus',
 'url': 'http://braininfo.rprc.washington.edu/brain-concept/2399/paracingulate-sulcus',
 'title': 'paracingulate sulcus - BrainInfo',
 'description': 'The term paracingulate sulcus refers to a cleft on the medial aspect of the superior frontal gyrus (SFG) that appears in approximately 35% of humans . Identifi…'}

In [ ]:
Harvard XML
    ↓

raw_brain_regions.json
(원본 보존)

    ↓

Qwen3
(메타데이터 생성)

    ↓

brain_regions_rag.json

    ↓

FAISS / ChromaDB

    ↓

QA

# 설문지에 사용할 각 영역별 요약 qwen 모델 사용

In [137]:
# 프롬프트 생성
def build_prompt(region):

    wiki_text = ""

    if region["wikipedia"]:
        wiki_text = region["wikipedia"].get(
            "text",
            ""
        )[:4000]

    braininfo_text = ""

    if region["braininfo"]:
        braininfo_text = region["braininfo"].get(
            "description",
            ""
        )

    return f"""
You are a neuroscientist.
Generate metadata for this brain region.
Choose at most 2 network tags.
Only include networks strongly supported by neuroscience literature.

Region:
{region['atlas_label']}

Wikipedia:
{wiki_text}

BrainInfo:
{braininfo_text}

Return ONLY JSON.

{{
  "overview": "",
  "anatomical_location": "",
  "major_functions": [],
  "network_tags": [],
  "functional_tags": [],
  "clinical_tags": []
}}

Rules:

network_tags must be chosen from:

[
"Default Mode Network",
"Salience Network",
"Central Executive Network",
"Dorsal Attention Network",
"Ventral Attention Network",
"Language Network",
"Sensorimotor Network",
"Visual Network",
"Auditory Network",
"Limbic Network"
]

functional_tags examples:

[
"Emotion",
"Memory",
"Language",
"Attention",
"Executive Function",
"Interoception",
"Motor Control",
"Vision",
"Audition",
"Decision Making",
"Social Cognition",
"Self Awareness"
]

clinical_tags examples:

[
"Anxiety",
"Depression",
"Addiction"
]
"""

In [138]:
def extract_json(text):

    if "</think>" in text:

        text = text.split(
            "</think>"
        )[-1]

    match = re.search(
        r'\{.*\}',
        text,
        re.DOTALL
    )

    if not match:
        return None

    return json.loads(
        match.group()
    )

In [139]:
# Ollama 호출
import ollama
import json

def summarize_region(region):

    prompt = build_prompt(region)

    response = ollama.chat(

        model="qwen3:14b",

        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    text = response["message"]["content"]

    return extract_json(text)

In [140]:
# insula 테스트
summary = summarize_region(
    raw_data[0]
)

print(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False
    )
)

{
  "overview": "The frontal pole is the most anterior region of the frontal lobe, part of the prefrontal cortex, and involved in higher-order cognitive functions, including executive control, decision-making, social cognition, and self-awareness.",
  "anatomical_location": "Located in the rostral portion of the frontal lobe, anterior to the central and lateral sulci, and within the prefrontal cortex.",
  "major_functions": [
    "Executive Function",
    "Decision Making",
    "Attention",
    "Working Memory",
    "Social Cognition",
    "Self Awareness",
    "Motivation",
    "Planning"
  ],
  "network_tags": [
    "Central Executive Network",
    "Salience Network"
  ],
  "functional_tags": [
    "Executive Function",
    "Decision Making",
    "Attention",
    "Working Memory",
    "Social Cognition",
    "Self Awareness"
  ],
  "clinical_tags": [
    "Addiction"
  ]
}


In [141]:
import json

with open(
    "raw_brain_regions.json",
    "r",
    encoding="utf-8"
) as f:
    raw_data = json.load(f)

results = []
failed_regions = []

for i, region in enumerate(raw_data):

    print(
        f"[{i+1}/{len(raw_data)}]Processing:",
        f"{region['atlas_label']}"
    )

    try:
        metadata = summarize_region(
            region
        )

    except Exception as e:

        print(
            "FAILED:",
            region["atlas_label"]
        )
        print(e)

        failed_regions.append(
            region["atlas_label"]
        )
        continue

    results.append({

        "index":
            region["index"],

        "atlas_label":
            region["atlas_label"],

        "metadata":
            metadata
    })

    with open(
        "brain_regions_rag.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            results,
            f,
            indent=2,
            ensure_ascii=False
        )

print("Done")
print(
    "Failed Regions:",
    failed_regions
)

[1/48]Processing: Frontal Pole
[2/48]Processing: Insular Cortex
[3/48]Processing: Superior Frontal Gyrus
[4/48]Processing: Middle Frontal Gyrus
[5/48]Processing: Inferior Frontal Gyrus, pars triangularis
[6/48]Processing: Inferior Frontal Gyrus, pars opercularis
[7/48]Processing: Precentral Gyrus
[8/48]Processing: Temporal Pole
[9/48]Processing: Superior Temporal Gyrus, anterior division
[10/48]Processing: Superior Temporal Gyrus, posterior division
[11/48]Processing: Middle Temporal Gyrus, anterior division
[12/48]Processing: Middle Temporal Gyrus, posterior division
[13/48]Processing: Middle Temporal Gyrus, temporooccipital part
[14/48]Processing: Inferior Temporal Gyrus, anterior division
[15/48]Processing: Inferior Temporal Gyrus, posterior division
[16/48]Processing: Inferior Temporal Gyrus, temporooccipital part
[17/48]Processing: Postcentral Gyrus
[18/48]Processing: Superior Parietal Lobule
[19/48]Processing: Supramarginal Gyrus, anterior division
[20/48]Processing: Supramargina

In [142]:
# 검증
for region in results:

    if not region["metadata"]["network_tags"]:

        print(region["atlas_label"])

In [ ]:
# 태그 빈호 확인
from collections import Counter

counter = Counter()

for region in results:

    counter.update(
        region["metadata"]["network_tags"]
    )

print(counter)

Counter({'Visual Network': 18, 'Salience Network': 17, 'Language Network': 14, 'Default Mode Network': 11, 'Sensorimotor Network': 6, 'Dorsal Attention Network': 6, 'Auditory Network': 5, 'Central Executive Network': 4, 'Limbic Network': 2, 'Ventral Attention Network': 1})


In [ ]:
# 언어 영역이 많이 나와서 확인해봄(무작위로 그냥 tag를 넣은거지 확인)
for region in results:

    if "Language Network" in region["metadata"]["network_tags"]:

        print(region["atlas_label"])

Inferior Frontal Gyrus, pars triangularis
Inferior Frontal Gyrus, pars opercularis
Superior Temporal Gyrus, anterior division
Superior Temporal Gyrus, posterior division
Middle Temporal Gyrus, anterior division
Middle Temporal Gyrus, posterior division
Middle Temporal Gyrus, temporooccipital part
Supramarginal Gyrus, anterior division
Supramarginal Gyrus, posterior division
Angular Gyrus
Frontal Operculum Cortex
Central Opercular Cortex
Planum Polare
Planum Temporale


In [ ]:
brain_regions_rag.json
(overview + tags)

        ↓
     임베딩

        ↓
      FAISS

        ↓
     질문 입력

        ↓
   Top-K 영역 검색

        ↓

brain_regions_rag.json
+
raw_brain_regions.json

        ↓

Qwen3

        ↓

최종 답변

In [7]:
# 임베딩
# 데이터 로드
import json

with open(
    "brain_regions_rag.json",
    "r",
    encoding="utf-8"
) as f:

    rag_data = json.load(f)

In [8]:
# 임베딩용 문서 생성(새로운 셀)
def make_document(region):

    meta = region["metadata"]

    return f"""
Region:
{region['atlas_label']}

Overview:
{meta['overview']}

Functions:
{', '.join(meta['major_functions'])}

Networks:
{', '.join(meta['network_tags'])}

Functional Tags:
{', '.join(meta['functional_tags'])}

Clinical Tags:
{', '.join(meta['clinical_tags'])}
"""

# 문서 생성
documents = []

for region in rag_data:

    documents.append(
        make_document(region)
    )

print(documents[0])


Region:
Frontal Pole

Overview:
The frontal pole is the most anterior region of the frontal lobe, part of the prefrontal cortex, and involved in higher-order cognitive functions such as goal-directed behavior, abstract thinking, and self-awareness.

Functions:
Executive Function, Decision Making, Social Cognition, Abstract Reasoning

Networks:
Central Executive Network, Default Mode Network

Functional Tags:
Executive Function, Decision Making, Social Cognition, Self Awareness

Clinical Tags:
Addiction, Impulse Control Disorders



In [9]:
# 임베딩 모델
from sentence_transformers import SentenceTransformer
#(faiss는 cpu버전을 설치함-RAG의 양이 적어서, 근데 임베팅은 gpu로 진행함)
embedding_model = SentenceTransformer(
    "intfloat/multilingual-e5-base",
    device="cuda"
)

In [10]:
# 임베팅 test
embedding_model.encode(
    ["hello"]
).shape # (1, 768)

(1, 768)

In [11]:
# 임베팅 생성
embeddings = embedding_model.encode(
    documents,
    normalize_embeddings=True,
    show_progress_bar=True
)
print(embeddings.shape) # (48, 768)

Batches: 100%|██████████| 2/2 [00:00<00:00,  3.93it/s]

(48, 768)


In [12]:
# FAISS 생성
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(
    dimension
)

index.add(
    np.array(
        embeddings,
        dtype=np.float32
    )
)

print(index.ntotal)

48


In [13]:
# FIASS와 임베딩 저장
faiss.write_index(
    index,
    "brain_regions.index"
)
np.save(
    "brain_embeddings.npy",
    embeddings
)

In [21]:
# 검색 함수
def search_regions(
    question,
    k=5
):

    query_embedding = embedding_model.encode(

        [question],

        normalize_embeddings=True
    )

    scores, ids = index.search(
        np.array(
            query_embedding,
            dtype=np.float32
        ),
        k
    )

    results = []

    for score, idx in zip(
        scores[0],
        ids[0]
    ):

        results.append({

            "score": float(score),

            "region":
            rag_data[idx]["atlas_label"]
        })

    return results

In [22]:
# 검색 테스트
search_regions(
    "emotion regulation"
)


[{'score': 0.8408060073852539,
  'region': 'Cingulate Gyrus, posterior division'},
 {'score': 0.8387079834938049, 'region': 'Cingulate Gyrus, anterior division'},
 {'score': 0.8335466384887695, 'region': 'Subcallosal Cortex'},
 {'score': 0.8326749801635742, 'region': 'Insular Cortex'},
 {'score': 0.831284761428833, 'region': 'Frontal Medial Cortex'}]

In [23]:
search_regions(
    "감정조절"
)

[{'score': 0.7493851184844971,
  'region': 'Middle Temporal Gyrus, posterior division'},
 {'score': 0.7460296154022217, 'region': 'Cingulate Gyrus, anterior division'},
 {'score': 0.7363346815109253,
  'region': 'Middle Temporal Gyrus, temporooccipital part'},
 {'score': 0.7361631393432617, 'region': 'Subcallosal Cortex'},
 {'score': 0.7359643578529358, 'region': 'Frontal Orbital Cortex'}]

In [1]:
import os
import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
import ollama

# ==========================================
# 1. 파일 로드 및 환경 설정
# ==========================================
print("🔄 JSON 데이터 및 FAISS 리소스를 로드 중...")

# [체크] 실제 환경에 맞게 두 json 파일 경로를 로드하세요.
with open("brain_regions_rag.json", "r", encoding="utf-8") as f:
    rag_json = json.load(f)  # 요약 및 태그 정보 데이터 ({ "Frontal Pole": {...}, "Superior Temporal Gyrus...": {...} } 형태라고 가정)

with open("raw_brain_regions.json", "r", encoding="utf-8") as f:
    raw_json = json.load(f)  # 위키피디아 원문 리스트 데이터 ([ {"atlas_label": "...", "wikipedia": {...}}, ... ])

# multilingual모델(영어, 한국어 다양) 임베딩 및 FAISS 로드
embedding_model = SentenceTransformer("intfloat/multilingual-e5-base", device="cuda")
current_dir = os.getcwd()
index = faiss.read_index(os.path.join(current_dir, "brain_regions.index"))

print("✅ 모든 데이터셋 로드 완료!")

/home/lhe09/miniconda3/envs/second/lib/python3.9/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/home/lhe09/miniconda3/envs/second/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔄 JSON 데이터 및 FAISS 리소스를 로드 중...
✅ 모든 데이터셋 로드 완료!


In [2]:
def generate_patient_report(tumor_data, top_regions_list):
    """
    top_regions_list: ["Superior Temporal Gyrus, anterior division", "Parahippocampal Gyrus, posterior division", ...]
    """
    # 1. 탑 5 영역에 해당하는 요약 정보를 rag_json에서 직접 추출
    extracted_infos = []
    for region_name in top_regions_list:
        # json 키값에 맞게 매칭 (딕셔너리 형태일 경우)
        if region_name in rag_json:
            info = rag_json[region_name]
            summary_text = (
                f"- 영역명: {region_name}\n"
                f"  개요: {info.get('overview', '')}\n"
                f"  주요기능: {', '.join(info.get('major_functions', []))}\n"
                f"  임상태그/증상: {', '.join(info.get('clinical_tags', []))}\n"
            )
            extracted_infos.append(summary_text)
        else:
            # 이름이 완벽히 일치하지 않을 때를 대비한 예외처리
            extracted_infos.append(f"- 영역명: {region_name} (상세 요약 정보 없음)")

    regions_summary_context = "\n".join(extracted_infos)

    # 2. Ollama를 통해 결과 리포트 문서화
    system_prompt = """당신은 환자의 뇌 종양 분석 결과를 기반으로 따뜻하고 이해하기 쉬운 '진단 결과지'를 작성하는 신경외과 AI 의사입니다.
제공되는 구조화된 뇌 영역 정보를 바탕으로 환자가 자신의 상태를 쉽게 이해할 수 있도록 한국어로 정중하게 작성해 주세요."""

    user_prompt = f"""
[환자 Tumor 분석 정보]
{tumor_data}

[종양 발견 영역들의 상세 기능 요약]
{regions_summary_context}

위 정보를 바탕으로 환자용 '종합 뇌 분석 결과지'를 한국어로 작성해 주세요.
반드시 포함할 내용:
1. 종양의 물리적 정보 (위치, 부피 등) 요약
2. 종양이 위치한 뇌 영역들이 원래 담당하던 역할(기능) 설명
3. 이 부위의 영향으로 인해 환자가 일상에서 마주할 수 있는 잠재적 증상(인지/감정적 변화) 설명
4. 환자와 보호자를 위한 따뜻한 위로문구식 조언
5. 정확한 증상 발현 여부와 치료 계획은 반드시 담당 주치의 선생님과 상의하셔야 한다고 꼭 넣어줘
"""

    response = ollama.chat(
        model="qwen3:14b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    return response["message"]["content"]

In [ ]:
# version 1
def ask_question_about_report(patient_hangul_question, tumor_data):
    query = patient_hangul_question
    
    # 2. FAISS 밀집 벡터 검색
    query_vector = embedding_model.encode([query])
    distances, indices = index.search(query_vector, k=5)  # 가장 연관 깊은 5개 영역 추출
    print(indices)
    for d, idx in zip(distances[0], indices[0]):
        print(raw_json[idx]["atlas_label"], d)

    # 3. raw_brain_regions.json에서 '위키피디아 원문' 및 'BrainInfo' 원문 추출
    rag_raw_contexts = []
    for idx in indices[0]:
        print(f"🔍 [QA 검색] 변환된 영어 검색어:", raw_json[idx]['atlas_label'])
        target_data = raw_json[idx]  # 리스트 인덱스로 접근
        atlas_label = target_data.get("atlas_label", "Unknown")
        wiki_text = target_data.get("wikipedia" or {}).get("text", "No Wikipedia content available.")
        braininfo_desc = target_data.get("braininfo" or {}).get("description", "")
        
        # QA 신뢰도를 높이기 위해 raw json의 방대한 텍스트와 메타 데이터를 가공
        #context_chunk = f"=== Brain Region: {atlas_label} ===\n[Wikipedia info]:\n{wiki_text[:2000]}\n[BrainInfo]: {braininfo_desc}" # 너무 길면 잘라줌
        metadata = rag_json[idx]["metadata"]
        context_chunk = f"""
        Brain Region:
        {atlas_label}

        Overview:
        {metadata.get("overview","")}

        Functions:
        {', '.join(metadata.get("major_functions",[]))}

        Clinical:
        {', '.join(metadata.get("clinical_tags",[]))}
        """
        rag_raw_contexts.append(context_chunk)
        
    full_raw_context = "\n\n".join(rag_raw_contexts)

    # 4. Ollama를 통해 방대한 위키 원문을 기반으로 한 팩트 기반 한국어 답변 생성
    system_prompt = """당신은 환자의 추가 질문에 답변하는 전문 AI 의사입니다.
제공되는 '원문 데이터(Wikipedia/BrainInfo)'에 철저히 기반하여 답변하되, 환자가 불안해하지 않도록 한국어로 명확하고 친절하게 설명해야 합니다.
답변 시 참고한 뇌 영역 이름(Atlas Label)을 명시해 주고, 최종 진단은 반드시 주치의와 상의하라는 코멘트를 추가하세요. 
본 답변은 의학 연구 데이터를 기반으로 한 참고용이므로, 정확한 증상 발현 여부와 치료 계획은 반드시 담당 주치의 선생님과 상의하셔야 합니다. 라는 문구는 고정으로 들어가게 해주세요"""

    user_prompt = f"""
[기본 환자 종양 상태]
{tumor_data}

[참고용 의학 원문 데이터 (영문)]
{full_raw_context}

[환자의 질문]
{patient_hangul_question}

위 의학 원문 데이터를 바탕으로, 환자의 질문에 정확한 근거를 들어 한국어로 답변해 주세요.
"""

    response = ollama.chat(
        model="qwen3:14b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    return response["message"]["content"]

In [ ]:
# 1. 분석 결과 서식 정의
msd_tumor_output = """
===== Tumor Analysis =====
Center (MNI mm): [ 20.73718732 -30.79664615 -36.98323073]
Location: Right Occipital (Inferior)
Tumor volume: 240.74 ml
Top Regions:
1. Superior Temporal Gyrus, anterior division (18.2%)
2. Parahippocampal Gyrus, posterior division (16.8%)
3. Temporal Occipital Fusiform Cortex (14.9%)
4. Temporal Fusiform Cortex, anterior division (11.5%)
5. Lingual Gyrus (7.9%)
"""

# 결과지 생성에 쓰일 상위 5개 영역 리스트
patient_top_regions = [
    "Superior Temporal Gyrus, anterior division",
    "Parahippocampal Gyrus, posterior division",
    "Temporal Occipital Fusiform Cortex",
    "Temporal Fusiform Cortex, anterior division",
    "Lingual Gyrus"
]

print("\n=============================================")
print("📋 1단계: 환자 맞춤형 분석 결과지 생성 중...")
print("=============================================")
patient_report = generate_patient_report(msd_tumor_output, patient_top_regions)
print(patient_report)

print("\n==================================================")
print("💬 2단계: 결과지 기반 실시간 사후 QA 챗봇 가동")
print("  - 종료하시려면 '종료', 'exit', 'q'를 입력하세요.")
print("==================================================")

while True:
    # 1. 터미널로부터 실시간 사용자 입력 받기
    patient_question = input("\n[환자/보호자 질문 입력] > ")
    
    # 2. 종료 조건 체크
    if patient_question.strip() in ["종료", "exit", "q", "quit"]:
        print("\n👋 실시간 QA 챗봇 서비스를 종료합니다. 쾌유를 빕니다.")
        break
        
    # 공백 입력 예외 처리
    if not patient_question.strip():
        print("질문 내용을 입력해 주세요.")
        continue
        
    print("\n🤖 AI가 의학 지식 원문(RAG)을 검색하여 답변을 준비 중입니다...")
    
    try:
        # 3. 앞서 만든 RAG 기반 QA 함수 호출
        # (내부적으로 한글->영어 변환 / FAISS 검색 / raw_json 위키 텍스트 취합 / Qwen3 생성이 실시간으로 돔)
        qa_answer = ask_question_about_report(patient_question, msd_tumor_output)
        
        # 4. 실시간 답변 출력
        print("\n[AI 의사의 상세 QA 답변]:")
        print("-" * 50)
        print(qa_answer)
        print("-" * 50)
        
    except Exception as e:
        print(f"\n❌ 답변 생성 중 오류가 발생했습니다: {e}")

In [14]:
# version 2
# 질문과 비슷한 뇌영역을 찾음
def get_symptom_regions(question):
    query_vector = embedding_model.encode([question])
    distances, indices = index.search(query_vector,k=5)
    regions = []
    for score, idx in zip(distances[0],indices[0]):
        regions.append({
            "atlas_label": rag_json[idx]["atlas_label"],
            "score": float(score),
            "metadata": rag_json[idx]["metadata"]
        })
    return regions

# 환자의 뇌영역 저장
def get_patient_regions(patient_top_regions):
    results = []
    for region_name in patient_top_regions:
        for item in rag_json:
            if (item["atlas_label"]== region_name):
                results.append(item)
                break
    return results

# 환자의 뇌영역과 질문 증상의 뇌영역을 비교
def compute_overlap(symptom_regions, patient_regions):
    symptom_names = {r["atlas_label"] for r in symptom_regions}
    patient_names = {r["atlas_label"] for r in patient_regions}
    overlap = list(symptom_names & patient_names)
    return overlap

# 질문의 키워드 정규화
def normalize_question(question):
    prompt = f"""
사용자의 증상을
1~3개의 뇌기능 키워드로 변환해라.

질문:
{question}

JSON 형식:
["keyword1","keyword2"]
"""

    response = ollama.chat(
        model="qwen3:14b",
        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ]
    )

    return response["message"]["content"]

def ask_question_about_report(patient_hangul_question, tumor_data, patient_top_regions):
    query = normalize_question(patient_hangul_question)

    symptom_regions = get_symptom_regions(patient_hangul_question)
    patient_regions = get_patient_regions(patient_top_regions)
    overlap = compute_overlap(symptom_regions,patient_regions)
    
    # raw_brain_regions.json에서 '위키피디아 원문' 및 'BrainInfo' 원문 추출
    rag_raw_contexts = []
    for region in symptom_regions:
        print(f"🔍 [QA 검색] 변환된 영어 검색어:", region['atlas_label'])
        atlas_label = region["atlas_label"]
        target_data = next((x for x in raw_json if x["atlas_label"] == atlas_label), None)
        if target_data is None : 
            continue
        wiki = target_data.get("wikipedia") or {}
        braininfo = target_data.get("braininfo") or {}
        wiki_text = wiki.get("text", "No Wikipedia content available.")
        braininfo_desc = braininfo.get("description", "")
        
        # QA 신뢰도를 높이기 위해 raw json의 방대한 텍스트와 메타 데이터를 가공
        #context_chunk = f"=== Brain Region: {atlas_label} ===\n[Wikipedia info]:\n{wiki_text[:2000]}\n[BrainInfo]: {braininfo_desc}" # 너무 길면 잘라줌
        metadata = region["metadata"]
        context_chunk = f"""
        Brain Region:
        {atlas_label}

        Overview:
        {metadata.get("overview","")}

        Functions:
        {', '.join(metadata.get("major_functions",[]))}

        Clinical:
        {', '.join(metadata.get("clinical_tags",[]))}
        """
        rag_raw_contexts.append(context_chunk)
        
    full_raw_context = "\n\n".join(rag_raw_contexts)

    # 4. Ollama를 통해 방대한 위키 원문을 기반으로 한 팩트 기반 한국어 답변 생성
    system_prompt = """당신은 환자의 추가 질문에 답변하는 전문 AI 의사입니다.
제공되는 '원문 데이터(Wikipedia/BrainInfo)'에 철저히 기반하여 답변하되, 환자가 불안해하지 않도록 한국어로 명확하고 친절하게 설명해야 합니다.
답변 시 참고한 뇌 영역 이름(Atlas Label)을 명시해 주고, 최종 진단은 반드시 주치의와 상의하라는 코멘트를 추가하세요. 
본 답변은 의학 연구 데이터를 기반으로 한 참고용이므로, 정확한 증상 발현 여부와 치료 계획은 반드시 담당 주치의 선생님과 상의하셔야 합니다. 라는 문구는 고정으로 들어가게 해주세요"""

    user_prompt = f"""
[환자 질문]

{patient_hangul_question}

================================

[일반적으로 관련된 뇌 영역]

{json.dumps(symptom_regions, ensure_ascii=False, indent=2)}

================================

[현재 환자의 종양 침범 영역]

{json.dumps(patient_regions, ensure_ascii=False, indent=2)}

================================

[직접 일치하는 영역]

{overlap}

================================

답변 규칙

1.
환자 질문과 관련된 뇌 기능을 먼저 설명한다.

2.
일반적으로 관련되는 뇌 영역을 설명한다.

3.
현재 환자의 종양 위치와 비교한다.

4.
관련성 수준을 반드시 판단한다.

관련성:
- 높음
- 중간
- 낮음

중 하나를 선택

5.
왜 그렇게 판단했는지 설명한다.

6.
환자가 이해하기 쉽게 한국어로 작성한다.

7.
의학적 진단은 반드시 담당 의료진과 상의해야 함을 명시한다.
"""

    response = ollama.chat(
        model="qwen3:14b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    return response["message"]["content"]

In [16]:
print("\n==================================================")
print("💬 2단계: 결과지 기반 실시간 사후 QA 챗봇 가동")
print("  - 종료하시려면 '종료', 'exit', 'q'를 입력하세요.")
print("==================================================")

msd_tumor_output = """
===== Tumor Analysis =====
Center (MNI mm): [ 20.73718732 -30.79664615 -36.98323073]
Location: Right Occipital (Inferior)
Tumor volume: 240.74 ml
Top Regions:
1. Superior Temporal Gyrus, anterior division (18.2%)
2. Parahippocampal Gyrus, posterior division (16.8%)
3. Temporal Occipital Fusiform Cortex (14.9%)
4. Temporal Fusiform Cortex, anterior division (11.5%)
5. Lingual Gyrus (7.9%)
"""

# 결과지 생성에 쓰일 상위 5개 영역 리스트
patient_top_regions = [
    "Superior Temporal Gyrus, anterior division",
    "Parahippocampal Gyrus, posterior division",
    "Temporal Occipital Fusiform Cortex",
    "Temporal Fusiform Cortex, anterior division",
    "Lingual Gyrus"
]

while True:
    # 1. 터미널로부터 실시간 사용자 입력 받기
    patient_question = input("\n[환자/보호자 질문 입력] > ")
    
    # 2. 종료 조건 체크
    if patient_question.strip() in ["종료", "exit", "q", "quit"]:
        print("\n👋 실시간 QA 챗봇 서비스를 종료합니다. 쾌유를 빕니다.")
        break
        
    # 공백 입력 예외 처리
    if not patient_question.strip():
        print("질문 내용을 입력해 주세요.")
        continue
        
    print("\n🤖 AI가 의학 지식 원문(RAG)을 검색하여 답변을 준비 중입니다...")
    
    try:
        # 3. 앞서 만든 RAG 기반 QA 함수 호출
        # (내부적으로 한글->영어 변환 / FAISS 검색 / raw_json 위키 텍스트 취합 / Qwen3 생성이 실시간으로 돔)
        qa_answer = ask_question_about_report(patient_question, msd_tumor_output, patient_top_regions)
        
        # 4. 실시간 답변 출력
        print("\n[AI 의사의 상세 QA 답변]:")
        print("-" * 50)
        print(qa_answer)
        print("-" * 50)
        
    except Exception as e:
        print(f"\n❌ 답변 생성 중 오류가 발생했습니다: {e}")


💬 2단계: 결과지 기반 실시간 사후 QA 챗봇 가동
  - 종료하시려면 '종료', 'exit', 'q'를 입력하세요.

🤖 AI가 의학 지식 원문(RAG)을 검색하여 답변을 준비 중입니다...
🔍 [QA 검색] 변환된 영어 검색어: Inferior Temporal Gyrus, anterior division
🔍 [QA 검색] 변환된 영어 검색어: Inferior Temporal Gyrus, posterior division
🔍 [QA 검색] 변환된 영어 검색어: Occipital Fusiform Gyrus
🔍 [QA 검색] 변환된 영어 검색어: Inferior Temporal Gyrus, temporooccipital part
🔍 [QA 검색] 변환된 영어 검색어: Middle Temporal Gyrus, posterior division

[AI 의사의 상세 QA 답변]:
--------------------------------------------------
**환자 질문에 대한 답변:**  

**1. 관련 뇌 기능 설명**  
사람의 얼굴을 인식하는 능력은 주로 **시각 정보 처리**와 **기억 회상**에 관여하는 뇌 영역에서 이루어집니다. 특히, 얼굴 인식은 **프루스페이스(Fusiform Face Area, FFA)**와 같은 구조가 포함된 **측두엽의 하부 측두 회장**(Inferior Temporal Gyrus) 및 **두정두회**(Occipital Fusiform Gyrus) 등의 영역에서 주로 처리됩니다. 이 영역이 손상되거나 기능이 저하되면 **얼굴 인식 장애**(Prosopagnosia)와 같은 증상이 나타날 수 있습니다.  

---

**2. 일반적으로 관련된 뇌 영역**  
- **측두엽 하부 회장(전측 분할, 후측 분할)**: 얼굴 인식, 색상 및 형태 처리, 기억 회상  
- **두정두회**(Occipital Fusiform Gyrus): 얼굴 및 물체 인식, 프루스페이스(FFA) 포함  
- **측두-두정 회**(Tempo